# `py-<PkgName>` — function-by-function R⇄Python parity dictionary

For users **migrating existing R code** to Python: every public R function in [`<UpstreamR>`](https://github.com/upstream/repo), with side-by-side R and Python calls on the same input and a parameter-by-parameter documentation table.

Follows the schema from [`omicverse-rebuildr/NOTEBOOKS.md §Notebook 3`](https://github.com/omicverse/omicverse-rebuildr/blob/main/NOTEBOOKS.md).

**How to use this notebook**:
1. Find the R function you want to translate in the table of contents.
2. Read the parameter table — every R parameter has a Python row.
3. Copy the Python code cell directly.
4. The numerical comparison below proves the output matches R.

**To re-execute**:
```bash
# 1. Refresh R outputs (one-time per fixture change):
conda activate $R_TEST_ENV
Rscript examples/r_per_function_dump.R

# 2. Re-run the notebook:
conda activate $PYTHON_TEST_ENV
jupyter nbconvert --to notebook --execute examples/function_by_function_R_parity.ipynb \
    --output function_by_function_R_parity.ipynb
```

## 1. Setup

In [ ]:
import os, subprocess, json, sys
for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[k] = '8'

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path('.').resolve()
PORT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'examples' else NOTEBOOK_DIR
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, 'omicverse-rebuildr/engine')
from parity_metrics import compute_parity

# Run the R dump once
R_DUMP = PORT_DIR / 'examples' / '_r_outputs'
if not R_DUMP.exists() or not list(R_DUMP.glob('*.json')):
    R_ENV = os.environ.get('R_TEST_ENV', '/path/to/your/R/env')
    subprocess.run(
        ['conda', 'run', '-p', R_ENV, 'Rscript', 'examples/r_per_function_dump.R'],
        check=True, cwd=PORT_DIR,
    )

def load_r(fname): return json.loads((R_DUMP / fname).read_text())

# Load Python-side fixture
# data = pyreadr.read_r(...) or similar
print('R outputs available:', [p.name for p in R_DUMP.glob('*.json')])

## 2. Function-by-function

Each subsection: parameter table → R one-liner (markdown) → Python equivalent (code) → numerical comparison.

### 2.1 `<r_fn_a>(<r_args>)` — <one-line description>

<one paragraph: what the function does>

**Parameters**:

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `data` | `data` | `matrix` / `DataFrame` | — | genes × cells | the input expression matrix |
| `param1` | `param1` | `int` | `9` | `≥ 1` | ... |
| `param2` | `param2` | `bool` | `TRUE` → `True` | — | ... |
| — | `seed` | `int` | `12345` | — | **new in Python** — matches R's internal `set.seed(12345)` |

**R one-liner** (markdown only — execution happens in `r_per_function_dump.R`):

```r
result_a <- <r_fn_a>(data, param1=9, param2=TRUE)
```

**Python equivalent**:

In [ ]:
# Run the Python version
# result_a = <pkgname>.<r_fn_a>(data, param1=9, param2=True)
# print(result_a)

**Numerical comparison vs R output**:

In [ ]:
# r_a = load_r('<r_fn_a>.json')
# metric = compute_parity(r_a['output1'], result_a, algorithm_class='deterministic')
# print(f'max abs err: {metric:.4e}  →  {"✅ exact" if metric < 1e-10 else "❌ diverges"}')

### 2.2, 2.3, ... — one subsection per public function

## 3. Aggregate verdict

| Function | Output | Class | Metric | Value | Pass |
|---|---|---|---|---|---|
| `<r_fn_a>` | `output1` | deterministic | max abs err | 0.0 | ✅ |
| `<r_fn_a>` | `output2` | clustering | ARI | 1.0000 | ✅ |
| `<r_fn_b>` | `output1` | ordinal | Pearson | 1.000000 | ✅ |
| ... | ... | ... | ... | ... | ... |

> Fill in this table from the per-function comparison cells above. The full
> verdict belongs in [`RECONSTRUCTION_REPORT.md §3.1`](../RECONSTRUCTION_REPORT.md).